# Jira Agent LangGraph Pipeline

Run these cells from top to bottom. Configure AWS credentials, Jira access, and GitHub access in your environment before executing the pipeline.

In [ ]:
import os
from typing import Any, TypedDict

from dotenv import load_dotenv
from botocore.config import Config
from langchain_aws import ChatBedrockConverse
from langchain_core.messages import HumanMessage
from langchain_mcp_adapters.client import MultiServerMCPClient
from langgraph.graph import END, START, StateGraph
from langgraph.prebuilt import create_react_agent
from pprint import pprint

load_dotenv(override=True)

In [ ]:
class GraphState(TypedDict):
    ticket_id: str
    jira_cloud_id: str
    repo_name: str
    jira_details: str
    github_context: str
    subtasks: str
    created_subtasks: str

In [ ]:
def build_llm() -> ChatBedrockConverse:
    model_id = os.getenv("BEDROCK_MODEL_ID")
    region_name = os.getenv("BEDROCK_REGION", "us-east-1")
    if not model_id:
        raise ValueError(
            "Missing required environment variable: BEDROCK_MODEL_ID. "
            "Set it to a Bedrock inference profile ID or ARN that your account can invoke."
        )
    return ChatBedrockConverse(
        model=model_id,
        region_name=region_name,
        temperature=0,
        config=Config(read_timeout=300),  # 5 minutes instead of default 60 seconds
    )

In [ ]:
def remote_mcp_config(
    url: str,
    token: str | None,
    transport: str,
    auth_scheme: str = "Bearer",
) -> dict[str, Any]:
    config: dict[str, Any] = {"url": url, "transport": transport}
    if token:
        config["headers"] = {"Authorization": f"{auth_scheme} {token}"}
    return config


def require_env(name: str) -> str:
    value = os.getenv(name)
    if not value:
        raise ValueError(f"Missing required environment variable: {name}")
    return value


def build_mcp_client() -> MultiServerMCPClient:
    servers = {
        "jira": remote_mcp_config(
            require_env("JIRA_MCP_URL"),
            os.getenv("JIRA_MCP_AUTH_TOKEN"),
            os.getenv("JIRA_MCP_TRANSPORT", "streamable_http"),
            os.getenv("JIRA_MCP_AUTH_SCHEME", "Basic"),
        ),
        "github": remote_mcp_config(
            require_env("GITHUB_MCP_URL"),
            os.getenv("GITHUB_MCP_AUTH_TOKEN"),
            os.getenv("GITHUB_MCP_TRANSPORT", "streamable_http"),
            os.getenv("GITHUB_MCP_AUTH_SCHEME", "Bearer"),
        ),
    }
    return MultiServerMCPClient(servers)

In [ ]:
JIRA_READ_TOOL_NAMES = {
    "getJiraIssue",
    "searchJiraIssuesUsingJql",
    "getTeamworkGraphContext",
    "getTeamworkGraphObject",
}

JIRA_SUBTASK_TOOL_NAMES = {
    "getJiraIssue",
    "getJiraProjectIssueTypesMetadata",
    "getJiraIssueTypeMetaWithFields",
    "createJiraIssue",
}

GITHUB_TOOL_NAMES = {
    "get_file_contents",
    "search_code",
    "search_repositories",
}

def select_tools(tools, allowed_names: set[str]):
    available_names = {tool.name for tool in tools}
    missing_names = sorted(allowed_names - available_names)
    if missing_names:
        raise ValueError(
            "Missing expected MCP tools: "
            f"{', '.join(missing_names)}. "
            f"Available tools: {', '.join(sorted(available_names))}"
        )
    return [tool for tool in tools if tool.name in allowed_names]

def build_workflow(llm: ChatBedrockConverse, jira_tools, github_tools, jira_subtask_tools):
    jira_agent = create_react_agent(
        llm,
        jira_tools,
        prompt=(
    "You are a Jira story retrieval agent. Your ONLY job is to fetch a Jira issue and return its raw content. "
    "STRICT RULES — VIOLATING ANY OF THESE IS NOT ALLOWED:\n"
    "1. Make ONLY ONE tool call per turn. Wait for each result before the next call.\n"
    "2. Your final response MUST contain ONLY the formatted Jira story. "
    "   Do NOT include any thinking process, reasoning steps, tool call logs, "
    "   planning text, 'I will now...', 'Let me...', XML tags, <thinking> blocks, "
    "   or any text that is not part of the Jira story itself.\n"
    "3. Do NOT add analysis, implementation advice, suggestions, or commentary of any kind.\n"
    "4. Do NOT invent or assume any field that is not in the fetched data.\n"
    "5. If a field is absent, write exactly: Not provided.\n"
    "6. Copy field values VERBATIM from the fetched Jira data. "
    "   Do NOT paraphrase, summarize, reword, or shorten any field. "
    "   The Description, Acceptance Criteria, and Comments must be copied character-for-character exactly as they appear in the ticket. "
    "   Never add words, remove words, or change any phrasing.\n\n"
    "FORMAT YOUR RESPONSE EXACTLY AS THESE MARKDOWN SECTIONS AND NOTHING ELSE:\n"
    "### Title\n"
    "### Issue Key\n"
    "### Status\n"
    "### Type\n"
    "### Reporter\n"
    "### Assignee\n"
    "### Priority\n"
    "### Description\n"
    "### Acceptance Criteria\n"
    "### Linked Issues\n"
    "### Comments\n"
    "### Attachments\n"
    "### Other Relevant Fields\n\n"
    "Your entire response must be ONLY these sections filled with real data from the Jira ticket. Nothing before, nothing after."
),


    )

    github_agent = create_react_agent(
        llm,
        github_tools,
        prompt=(
    "You are a codebase analysis agent. Your job is to inspect a repository and report ONLY what you actually see. "
    "You will receive Jira story content as input. First, understand the story requirements, acceptance criteria, scope, "
    "and constraints from that Jira story. Then use those story details to decide what parts of the repository to inspect. "
    "Your goal is to map the Jira story to the exact relevant code paths, pages, APIs, models, schemas, queries, and tests in the repo. "
    "You are NOT allowed to guess, assume, infer, or invent ANYTHING. "
    "Every single claim in your final answer must be backed by code you literally read.\n\n"

    "MANDATORY SEQUENCE — follow this exactly, ONE tool call per turn:\n"
    "TURN 1: Call get_file_contents on 'README.md' ONLY. Do not call anything else. "
    "After reading it, identify the exact tech stack: language, framework, database type, route prefix, folder structure.\n"
    "TURN 2: Call get_file_contents with path '' to list the root directory ONLY.\n"
    "TURN 3+: Call get_file_contents on subdirectories one at a time (e.g. 'backend/', 'frontend/'). "
    "Only explore paths you saw in a previous directory listing. Never guess a path.\n"
    "TURN N+: Call get_file_contents on specific implementation files one at a time. "
    "Only open files whose exact path you saw in a directory listing. Prioritize files that are directly related to the Jira story requirements.\n\n"

    "ABSOLUTE RULES — breaking any of these is not allowed:\n"
    "1. ONE tool call per turn. Never batch multiple tool calls.\n"
    "2. Never call get_file_contents on a path you have not seen in a directory listing.\n"
    "3. Never mention a file in your final answer unless you called get_file_contents on it and read its contents.\n"
    "4. Never guess a database type, column name, table name, model field, or schema. "
    "   Report only field names and types you literally saw in a model/schema file.\n"
    "5. Never guess a route prefix (e.g. /api/v1/). Use only the prefix you saw in the actual route files.\n"
    "6. Never suggest adding tests if you did not find a test directory or test framework in the repo.\n"
    "7. Never reuse a component, function, or utility unless you saw it in the actual code you read.\n"
    "8. If you did not find a file, say so explicitly under Open Codebase Questions. Do not invent what it probably contains.\n"
    "9. Search broadly enough to cover the actual story surface area. Do not stop after the first matching file if the Jira story clearly implies related frontend, backend, API, schema, data, or test changes.\n"
    "10. Do not include <thinking> tags, tool call logs, planning text, or future inspection plans.\n\n"

    "FINAL ANSWER FORMAT — use these exact sections:\n"
    "### Relevant Files\n"
    "List only files you actually read. For each: state the exact path, what you found inside, and why it matters.\n\n"
    "### Existing Behavior\n"
    "Describe only behavior you saw in actual code. Quote relevant lines or field names directly from the files you read.\n\n"
    "### Required Implementation\n"
    "Describe only changes grounded in the actual code you read. Use exact field names, route prefixes, "
    "and patterns already present in the codebase, and tie each proposed change back to the Jira story requirements.\n\n"
    "### Integration Points\n"
    "Describe only connections between components you actually inspected.\n\n"
    "### Data/Config Changes\n"
    "List only changes to fields, models, or config you confirmed exist in the actual files.\n\n"
    "### Tests to Add or Update\n"
    "Only include this section if you found an existing test directory and test framework in the repo. "
    "If no tests exist, write: No test framework found in this repository.\n\n"
    "### Open Codebase Questions\n"
    "List every file or detail you could not confirm because you did not find or read the relevant file."
),



    )

    create_subtask_agent = create_react_agent(
        llm,
        jira_subtask_tools,
        prompt=(
            "You are a Jira subtask creation agent. Your ONLY job is to create Jira subtasks under "
            "the provided parent story by using the exact `subtasks` content passed to you. The "
            "provided `subtasks` text is the single source of truth. You must strictly refer only "
            "to that content when creating subtasks. Do not rewrite, expand, summarize, merge, "
            "split, reorder, paraphrase, or improve the provided subtasks. Do not invent any new "
            "subtask, title, detail, file, acceptance check, or requirement. Create only Jira "
            "Sub-task issues under the provided parent story; never create stories, tasks, bugs, "
            "comments, transitions, links, or parent edits. Make exactly ONE tool call per turn "
            "and wait for its result before making the next tool call. Never batch multiple tool "
            "calls together. First fetch the parent issue with "
            "getJiraIssue to confirm the project. Use getJiraProjectIssueTypesMetadata only if "
            "needed to confirm the exact Jira subtask issue type name. Use getJiraIssueTypeMetaWithFields "
            "only if needed to confirm required creation fields. Then call createJiraIssue once per "
            "provided subtask with cloudId, projectKey, issueTypeName, summary, description, parent, "
            "and contentFormat='markdown'. Use issueTypeName='Sub-task' unless Jira metadata shows a "
            "different subtask issue type name. Set parent to the provided parent story key. Use the "
            "exact text after '- Subtask:' as the Jira summary, unchanged. In the Jira description, "
            "preserve the provided Details, Files/Areas, and Acceptance Check content as written; do "
            "not add any extra commentary. Your final response must contain only a Markdown list of "
            "the created Jira subtask keys and their exact summaries. If creation of a subtask fails, "
            "return that subtask title and the exact Jira error. Do not include thinking, tool logs, "
            "planning text, or any content outside the final created-subtask result list."
        ),
    )

    async def fetch_jira(state: GraphState):
        print(f"--> [Jira Agent] Fetching details for {state['ticket_id']}...")
        prompt = (
            f"Fetch the Jira story using Jira MCP tools with these exact identifiers:\n"
            f"cloudId: {state['jira_cloud_id']}\n"
            f"objectType: JiraWorkItem\n"
            f"objectIdentifier: {state['ticket_id']}\n\n"
            f"Use getTeamworkGraphContext if connected context is needed, then use "
            f"getTeamworkGraphObject to fetch the full Jira work item content. "
            f"Return only the formatted Jira story content requested in your system prompt."
        )
        response = await jira_agent.ainvoke(
          {"messages": [("user", prompt)]},
            config={"recursion_limit": 30, "max_concurrency": 1}
        )

        pprint(response)
        return {"jira_details": response["messages"][-1].content}

    async def fetch_github(state: GraphState):
        print("--> [GitHub Agent] Searching codebase...")
        prompt = (
            f"Repository: {state['repo_name']}\n\n"
            f"Jira story content:\n{state['jira_details']}\n\n"
            f"First understand the Jira story requirements, acceptance criteria, and scope. Then use "
            f"GitHub MCP tools to inspect this repository thoroughly based on those story details. "
            f"Read README.md first when it exists. Then inspect the relevant implementation files, "
            f"not just the first matching file. Search for related frontend files, backend/API files, "
            f"schemas/models, data-fetching code, queries, and tests needed to map this story to the codebase. "
            f"Return only the final formatted codebase context. Do not include <thinking> tags, "
            f"tool narration, or statements about future inspection."
        )
        response = await github_agent.ainvoke(
            {"messages": [("user", prompt)]},
            config={"recursion_limit": 200, "max_concurrency": 1})

        pprint(response)
        return {"github_context": response["messages"][-1].content}

    async def plan_subtasks(state: GraphState):
        print("--> [Planner Agent] Generating final subtasks...")
        prompt = (
            f"You are a lead developer. Break this Jira story down into subtasks.\n\n"
            f"JIRA STORY:\n{state['jira_details']}\n\n"
            f"CODEBASE CONTEXT:\n{state['github_context']}\n\n"
            f"Return only technical subtasks. Do not include introductions, summaries, notes, "
            f"assumptions, or any text outside the subtasks. Create only meaningful implementation "
            f"chunks that could reasonably be assigned or completed as distinct pieces of work. "
            f"Do not split small, tightly related changes into separate subtasks when they belong "
            f"to the same file, component, endpoint, or workflow. Merge closely related UI changes "
            f"into one frontend subtask, merge closely related schema/API changes into one backend "
            f"subtask, and keep tests with the area they validate unless test work is substantial. "
            f"Only create a separate subtask when the work touches a clearly separate layer, module, "
            f"or ownership boundary. Format each subtask exactly as:\n"
            f"- Subtask: <short action-oriented title>\n"
            f"  Details: <specific implementation work>\n"
            f"  Files/Areas: <relevant files or code areas>\n"
            f"  Acceptance Check: <how to verify completion>\n"
            f"Only include subtasks that are supported by the Jira story and codebase context. "
            f"Do not invent requirements."
        )
        response = await llm.ainvoke([HumanMessage(content=prompt)])
        return {"subtasks": response.content}

    async def create_subtasks(state: GraphState):
        print("--> [Jira Subtask Agent] Creating subtasks in Jira...")
        project_key = state["ticket_id"].split("-", 1)[0]
        prompt = (
            f"Create Jira subtasks under the provided parent story using ONLY the exact `subtasks` "
            f"content below as the source of truth.\n\n"
            f"cloudId: {state['jira_cloud_id']}\n"
            f"parentStoryKey: {state['ticket_id']}\n"
            f"projectKey: {project_key}\n\n"
            f"Subtasks to create exactly as provided:\n{state['subtasks']}\n\n"
            f"Create one Jira subtask per provided subtask block. Keep the summary and content faithful "
            f"to the provided text. Return only the created Jira subtask keys and exact summaries."
        )
        response = await create_subtask_agent.ainvoke(
            {"messages": [("user", prompt)]},
            config={"recursion_limit": 80, "max_concurrency": 1}
        )
        pprint(response)
        return {"created_subtasks": response["messages"][-1].content}

    workflow = StateGraph(GraphState)
    workflow.add_node("jira_node", fetch_jira)
    workflow.add_node("github_node", fetch_github)
    workflow.add_node("planner_node", plan_subtasks)
    workflow.add_node("create_subtasks_node", create_subtasks)

    workflow.add_edge(START, "jira_node")
    workflow.add_edge("jira_node", "github_node")
    workflow.add_edge("github_node", "planner_node")
    workflow.add_edge("planner_node", "create_subtasks_node")
    workflow.add_edge("create_subtasks_node", END)

    return workflow.compile()

In [ ]:
async def run_pipeline(ticket_id: str, repo_name: str):
    llm = build_llm()
    jira_cloud_id = require_env("JIRA_CLOUD_ID")
    mcp_client = build_mcp_client()

    jira_all_tools = await mcp_client.get_tools(server_name="jira")
    jira_tools = select_tools(jira_all_tools, JIRA_READ_TOOL_NAMES)
    jira_subtask_tools = select_tools(jira_all_tools, JIRA_SUBTASK_TOOL_NAMES)
    github_tools = select_tools(
        await mcp_client.get_tools(server_name="github"),
        GITHUB_TOOL_NAMES,
    )
    app = build_workflow(llm, jira_tools, github_tools, jira_subtask_tools)

    from IPython.display import Image, display

    display(Image(app.get_graph().draw_mermaid_png()))

    print("\nStarting the LangGraph Pipeline...\n")
    final_state = await app.ainvoke(
        {"ticket_id": ticket_id, "jira_cloud_id": jira_cloud_id, "repo_name": repo_name}
    )

    print("\n=== FINAL SUBTASKS ===")
    print(final_state["subtasks"])
    print("\n=== CREATED JIRA SUBTASKS ===")
    print(final_state["created_subtasks"])
    return final_state

In [ ]:
TICKET_ID = "<jira-ticket-id>"
REPO_NAME = "<github-repo-name>"

final_state = await run_pipeline(TICKET_ID, REPO_NAME)

In [ ]:

pprint(final_state)